In [1]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




[1] "Connected to : yhcr-prd-bradfor-bia-core"


In [3]:
person <- read.csv("data/person_cohorts_max.csv", header = TRUE)

# Mental Health Referrals 

In [2]:
MHref_data = "CB_2489.tbl_Mental_SRReferralIn"

MHref_table <- tbl(con, MHref_data) |>
    select(person_id, datereferral) |> 
    # convert to df
    collect()

In [4]:
head(person)

,person_id,birth_date,gender,CombinedEthnicity,cohort
,<chr>,<chr>,<chr>,<chr>,<chr>
1,EE91E3C77F2B7CB44F1F601317E2DE5EE76ABB0941F24D4714026B3E30FC313C,1998-09-15,M,South Asian,1998/99
2,C5553DE427E9B8F60E9B5D1E83391E1B660F415A3924B630742D52A0517F5219,1998-09-15,M,White British,1998/99
3,AC8CB4071E24E606BFFECC380A9EA11546158370916C4D8D954FA76D63FA81AE,1998-09-15,M,White British,1998/99
4,E4E84D0DEB6692712A8E875B701E097EB61235069C56626900C0951E9B112B6D,1998-09-15,M,South Asian,1998/99
5,64FA4602D88C275E27FDBD17EA4678E378D7B297D91C2CDB914150ED0CBF3785,1998-09-15,M,White British,1998/99
6,F516562D5CFDDC6CD66FB8EA18399D11BB0E2E042F569D918285FF96C3377338,1998-09-15,M,White British,1998/99


In [5]:
head(MHref_table)

person_id,datereferral
<chr>,<chr>
B63CC2FC864F90AF0CFCEEA3EF103E226EC91062EE704F022BACDEFCE1B93BEB,30 Jan 2018 11:34:11
B63CC2FC864F90AF0CFCEEA3EF103E226EC91062EE704F022BACDEFCE1B93BEB,30 Jan 2018 11:34:11
65DFC7AF06804B357EB3BA6D7A6205FD7BCF82AAF117E4323505C25F02EAE31B,20 Mar 2018 10:33:53
65DFC7AF06804B357EB3BA6D7A6205FD7BCF82AAF117E4323505C25F02EAE31B,20 Mar 2018 10:33:53
262B531379C8FABE17F50CA985D32ABA9715E0EDE59B2AC36E8F13ADDFDAD483,20 Mar 2018 10:57:35
262B531379C8FABE17F50CA985D32ABA9715E0EDE59B2AC36E8F13ADDFDAD483,20 Mar 2018 10:57:35


In [ ]:
intersect(person$person_id, MHref_table$person_id)

In [7]:
mh_filtered <- MHref_table |>
    # filter destinations to target person ids
    semi_join(person, by = "person_id")

In [9]:
mh_filtered <- mh_filtered |>
    unique()

In [11]:
head(mh_filtered)

person_id,datereferral
<chr>,<chr>
367B454437C575ADEB9D2E5BB831BC344B23F4ED34B6B4E35CF4EA1DF6C18230,30 Nov 2018 15:34:54
589529480A34074E589066C15276888E113B3D26D05811BD60ABFA6A62373E6D,11 Jun 2019 15:55:25
880D672D5444ABF736A4CFFE02ABAB2054E4486D9D06E9C99A6BAABED73428B7,11 Jun 2019 16:01:04
284CBB5A7E06A6F8862C38F09C39FE1C1FCF9CBE203E203E1714752A217A51A1,11 Sep 2018 16:15:08
8EE74633BEAAF8B1102C43B0DAFE4387F829574B7F60FCE9C60DEF287911C55C,21 Nov 2018 13:53:45
03D7DF202BB46EF8910D372A3E82506445C452139B888BBCA29BFEAC630ED50C,18 Jan 2019 16:54:45


In [13]:
mh_filtered <- mh_filtered |>
    mutate(MHReferral = as.Date(datereferral, format = "%d %b %Y %H:%M:%S"))

In [14]:
mh_filtered

person_id,datereferral,MHReferral
<chr>,<chr>,<date>
367B454437C575ADEB9D2E5BB831BC344B23F4ED34B6B4E35CF4EA1DF6C18230,30 Nov 2018 15:34:54,2018-11-30
589529480A34074E589066C15276888E113B3D26D05811BD60ABFA6A62373E6D,11 Jun 2019 15:55:25,2019-06-11
880D672D5444ABF736A4CFFE02ABAB2054E4486D9D06E9C99A6BAABED73428B7,11 Jun 2019 16:01:04,2019-06-11
284CBB5A7E06A6F8862C38F09C39FE1C1FCF9CBE203E203E1714752A217A51A1,11 Sep 2018 16:15:08,2018-09-11
8EE74633BEAAF8B1102C43B0DAFE4387F829574B7F60FCE9C60DEF287911C55C,21 Nov 2018 13:53:45,2018-11-21
03D7DF202BB46EF8910D372A3E82506445C452139B888BBCA29BFEAC630ED50C,18 Jan 2019 16:54:45,2019-01-18
A750B01D5FDBBE5F5A9444EC11B0773438BEEC7A0E24B66BB018DFD0282A133D,28 Oct 2019 14:37:15,2019-10-28
637193BDB068DD712A04B058165E2876BA9C9D68FC1EEDA978304D46A026D97D,30 Oct 2019 13:56:03,2019-10-30
B9FDC8506D505035AFBC556C0DF197EC924E34C886E786F862FE65ADC36D3EBF,04 Dec 2019 15:27:00,2019-12-04


In [20]:
cohort_9899 <- person |>
    filter(cohort == '1998/99')

In [22]:
mh_filtered_9899 <- mh_filtered |>
    semi_join(cohort_9899, by = "person_id") |>
    filter(MHReferral < as.Date("2015-09-01"))

In [116]:
mh_filtered_9899 |>
    arrange(MHReferral)

person_id,datereferral,MHReferral
<chr>,<chr>,<date>
1EC171D9C8E9515AF835F585890726E1987E8A4F15A3546BA682A21F163F34E9,30 Jan 2003 00:00:00,2003-01-30
8CCA4D6C724BACE60A211E9BFF8895BFE6EC2F82C307C4CB7A830FA87E03D7C5,16 May 2005 00:00:00,2005-05-16
E5E013E5BBFDA1F263E1BCDB98A42E8C8368C9CE4A415A181944464CC086E5AD,01 Mar 2006 00:00:00,2006-03-01
063DADDF0002D93D659DED1DA5565FDBB3775BAD559F8A041F8F7B38EC02CBEB,30 Nov 2007 00:00:00,2007-11-30
E031AFDFF72CE03410622827DD6988A3A3FC5BC303962D6313E0DBEFD454470A,08 Jul 2009 00:00:00,2009-07-08
6CFF4F6D5DEAFA9717F323D464A1FB52A2BC4DCEAF3776716022474526571069,11 Nov 2009 00:00:00,2009-11-11
FB81092AD28CA901FE1E58F0F36F6FE5236B7755132F74E65403AC8C20C761F2,22 Apr 2010 00:00:00,2010-04-22
DF6B3FF6DB820B472CB54DB871E2DC78C73A33C7D433C113AB87A112AD814B05,28 Apr 2010 00:00:00,2010-04-28
173837E14BAF55207D72F0E441AD26577205FEE02250ADF16D43703D4B80D07E,24 Nov 2010 15:56:00,2010-11-24


In [25]:
mh_filtered_9899 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,5449
2,151
3,12


In [30]:
unique(person$cohort)

[1] "1998/99" "1999/00" "2000/01" "2001/02"

In [32]:
cohort_9900 <- person |>
    filter(cohort == '1999/00')

In [34]:
mh_filtered_9900 <- mh_filtered |>
    semi_join(cohort_9900, by = "person_id") |>
    filter(MHReferral < as.Date("2016-09-01"))

In [36]:
mh_filtered_9900 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,5791
2,319
3,54
4,10
5,1
6,1


In [37]:
mh_filtered_9900 |>
  count(person_id) |>
  filter(n > 4) |>
  arrange(desc(n)) 

person_id,n
<chr>,<int>
531F491C7677E11BF58DEC4BB5C2791781E2A7F6233446B77FAD9F2E4EAC31D7,6
42447B8B8A730AC62487C348B6906A2A13267A68320A608A07786F3296B9C4EB,5


In [38]:
cohort_0001 <- person |>
    filter(cohort == '2000/01')

In [39]:
mh_filtered_0001 <- mh_filtered |>
    semi_join(cohort_0001, by = "person_id") |>
    filter(MHReferral < as.Date("2017-09-01"))

In [40]:
mh_filtered_0001 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,5673
2,436
3,118
4,42
5,19
6,4
8,2


In [41]:
cohort_0102 <- person |>
    filter(cohort == '2001/02')

In [42]:
mh_filtered_0102 <- mh_filtered |>
    semi_join(cohort_0102, by = "person_id") |>
    filter(MHReferral < as.Date("2018-09-01"))

In [43]:
mh_filtered_0102 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,5745
2,531
3,213
4,54
5,19
6,11
8,2
10,1


In [45]:
mh_filtered_0102 |>
  count(person_id) |>
  filter(n > 9) |>
  arrange(desc(n)) 

person_id,n
<chr>,<int>
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,10


In [47]:
mh_filtered_0102 |>
    filter(person_id == '77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1')

person_id,datereferral,MHReferral
<chr>,<chr>,<date>
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,18 Jun 2018 15:02:32,2018-06-18
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,20 Dec 2017 11:26:00,2017-12-20
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,06 Aug 2018 08:25:19,2018-08-06
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,07 Jan 2014 10:30:30,2014-01-07
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,27 Jan 2017 10:59:00,2017-01-27
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,01 Jun 2017 09:58:00,2017-06-01
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,03 Feb 2017 14:36:00,2017-02-03
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,21 Mar 2016 09:35:00,2016-03-21
77C5067B347B3B69B9800CBF7F886ED570ACF607BC22B3D3E3D1FACCBA23F9F1,01 Jun 2017 10:31:00,2017-06-01


In [51]:
mh_referrals <- rbind(mh_filtered_9899, mh_filtered_9900, mh_filtered_0001, mh_filtered_0102) |>
    select(-datereferral)

In [52]:
head(mh_referrals)

person_id,MHReferral
<chr>,<date>
32454BB17C37F7065B3BCBF64C4D14B327B3D64C7AF28C0314469AFB12781E84,2014-09-10
E26E1F2A1F0D213B84FBBAD0A9F715C1FFF85507204FD7C6DC58E42D02FBE714,2014-08-18
7F9F55AADAE38F6F1E4B8F02E5051C00B81292EE1B0CA278E0871B72DE70A4AA,2014-08-18
5958C58EED04EB09709716F8FBB5ABF8220F0378137F829774ED4B23556BD19A,2014-09-05
B1F6E94ECCD2170D1207F595A99611869DB631191BBBFCB312AD2AA0034DF8EA,2014-01-23
0E0DEE39B051BEC9E5024AB99475C6E2F50DC4B9B71505AA8BE421293EC1BB24,2014-07-18


In [53]:
write.csv(mh_referrals, "data/mh_referrals.csv", row.names = FALSE)

# Inpatient Ward Admissions

In [54]:
inpatient_data = "CB_2489.tbl_SecondaryCare_ward_stay"

inpatient_table <- tbl(con, inpatient_data) |>
    select(person_id, stay_start_date, length_of_stay) |> 
    # convert to df
    collect()

Auto-refreshing stale OAuth token.



In [57]:
inpatient_filtered <- inpatient_table |>
    # filter destinations to target person ids
    semi_join(person, by = "person_id") |>
    unique()

In [115]:
inpatient_filtered |>
    arrange(stay_start_date)

person_id,stay_start_date,length_of_stay
<chr>,<chr>,<dbl>
F463ADF06E5E74F420FA8B0CC64578C730B48D2A1BD10F22ED50FE938C42E21F,2006-12-29,3
92D116B37F94927C3909E3A0DEC93313C6E9DC8AE59A74E3741A4EC31BE423F6,2006-12-30,2
5DEB32EF0AC32EE350A65B88F618E29861012B390A6F8001BA406DA44788727B,2006-12-30,2
F7925D2EF45550630AC5551706DDD163DDC976AC8F14A2F68506BA3B346C9482,2006-12-31,1
F850A74E281BD8976AAC7660FC8012B54C19BA349AC227BBBCE205CE8B2EE470,2006-12-31,1
B9BA8CA19EB91125A2D67B96A3AD8C7FAFB3B9400C431D270F1747A5B2CB3021,2007-01-01,2
24A9A2DDFB03209798608768B0A03721528D0FF0E2156267BD7C0EED89A2CF92,2007-01-01,1
ABC8CFD844B24C7CF61AC7A0ADC1E6120FB060EA6A898DCAABE543D0D08C2A62,2007-01-01,3
622B00464CAC4229688514F6DE888BCE8EE2A93E3A74CB233E14E2F712953F10,2007-01-02,2


In [117]:
inpatient_filtered_9899 <- inpatient_filtered |>
    semi_join(cohort_9899, by = "person_id") |>
    filter(stay_start_date < as.Date("2015-09-01")) |>
    filter(stay_start_date >= as.Date("2007-09-01"))

In [118]:
inpatient_filtered_9899 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,1073
2,321
3,145
4,37
5,30
6,12
7,11
8,10
9,1


In [119]:
inpatient_filtered_9900 <- inpatient_filtered |>
    semi_join(cohort_9900, by = "person_id") |>
    filter(stay_start_date < as.Date("2016-09-01")) |>
    filter(stay_start_date >= as.Date("2008-09-01"))

In [120]:
inpatient_filtered_9900 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,1071
2,305
3,106
4,52
5,24
6,9
7,8
8,10
9,1


In [123]:
inpatient_filtered_0001 <- inpatient_filtered |>
    semi_join(cohort_0001, by = "person_id") |>
    filter(stay_start_date < as.Date("2017-09-01")) |>
    filter(stay_start_date >= as.Date("2009-09-01"))

In [122]:
inpatient_filtered_0001 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,1102
2,386
3,121
4,70
5,33
6,18
7,10
8,9
9,12


In [124]:
inpatient_filtered_0102 <- inpatient_filtered |>
    semi_join(cohort_0102, by = "person_id") |>
    filter(stay_start_date < as.Date("2018-09-01")) |>
    filter(stay_start_date >= as.Date("2010-09-01"))

In [125]:
inpatient_filtered_0102 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,1022
2,308
3,138
4,47
5,34
6,7
7,10
8,11
9,3


In [126]:
inpatient_stays <- rbind(inpatient_filtered_9899, inpatient_filtered_9900, inpatient_filtered_0001, inpatient_filtered_0102)

In [127]:
head(inpatient_stays)

person_id,stay_start_date,length_of_stay
<chr>,<chr>,<dbl>
01E2844C0F45245F1345B5AA5E94793B95F67AEE51E4469B1576CE4064799483,2007-09-01,2
D5EDDC33A87E4B5FAAE509E44B645A41DAB4DEF3D4B2B11775C39BEF1E3B79D5,2007-09-11,2
39C36EAA03BBF80F9A626ECD00FA24D6167311EA3EAA27CEFFE2B37D7817F6CC,2007-09-18,2
9D7BBB097EAD621D91C14965F614BD4573960C701384581B662362256879AF96,2007-09-19,2
A4324797DBF898025A89D6A94779F59EB3DC6D24607D051A75CDEA5B51B1D3F9,2007-09-24,2
1C0DE5273D202D08A0207914E83D87825D98FCEB618F46B2087CAFA032D09191,2007-10-01,2


In [128]:
write.csv(inpatient_stays, "data/inpatient_stays.csv", row.names = FALSE)

# A&E Admissions - Cant use, data from 2017 onwards

In [70]:
AE_data = "CB_2489.tbl_SecondaryCare_ae_nautilus"

AE_table <- tbl(con, AE_data) |>
    select(person_id, tbl_SecondaryCare_ae_nautilus_start_date, attendance_category_Description) |> 
    # convert to df
    collect()

Auto-refreshing stale OAuth token.



In [72]:
head(AE_table)

person_id,tbl_SecondaryCare_ae_nautilus_start_date,attendance_category_Description
<chr>,<chr>,<chr>
5AAAC46458805915BE03E39E2EE548428C3427A64D919B30D6F9E42C70C45044,2021-10-10 04:01:00,First A&E Attendance
5C1E6B4663EE26D02BE833DCAA4C0B8C7F78AFAD286830F58C8D41358C20C3BD,2021-11-05 05:00:00,First A&E Attendance
AF26FF8CE455CC63945131643FD33864D7B7017FE2E171C89DCEC7D0B24BFCFD,2023-04-24 22:51:00,First A&E Attendance
D6CDC2BB53B9F7A2F8945C63DF885FDE8549D31C47383F74AD7D6714CAD91F0C,2023-05-14 02:03:00,First A&E Attendance
EFBD06ECD8259E20EB320817CE841DE33E4E5C71CCE9CC8DDB1A2BD722E99E91,2024-01-01 03:00:00,First A&E Attendance
8C0B8CB3911D88B9344889562B7A05517AB18D1C418E51B3B957DDD32930EBE2,2024-02-10 01:21:00,First A&E Attendance


In [73]:
unique(AE_table$attendance_category_Description)

[1] "First A&E Attendance"             "Follow Up Attendance - Planned"  
[3] "Unplanned - F Up Another ED"      "Unplanned - F Up This ED"        
[5] "Follow Up Attendance - Unplanned" "DOA"

In [98]:
AE_filtered <- AE_table |>
    # filter destinations to target person ids
    semi_join(person, by = "person_id") |>
    unique()

In [114]:
AE_filtered |>
    arrange(tbl_SecondaryCare_ae_nautilus_start_date)

person_id,tbl_SecondaryCare_ae_nautilus_start_date,description,start_date
<chr>,<chr>,<chr>,<date>
936FEBEFFBDBA17B7FFDDE1D655BE365E0E45BB136C5536D760086339B63B543,2016-06-11 14:00:00,First A&E Attendance,2016-06-11
FA8F3FE65CEDDE1CAADBFF0F235E53AECC56EFBDCE6DA24CBDBF88A1F6B35063,2017-01-26 09:03:00,First A&E Attendance,2017-01-26
E27CDB12A315A24D1BDCCC488825B17495CEEB78C19F07059168EC79547FF53E,2017-02-05 14:00:00,First A&E Attendance,2017-02-05
FF0790E5C3F880E4CE48A311A2A9626393FCDD64E64C18D540F90A985C572BB7,2017-06-23 11:00:00,First A&E Attendance,2017-06-23
F23CEA945D0FC641A7565D9670C2B2C24BDE11F4759F5B5DF077262F4796DEE7,2017-06-25 18:00:00,First A&E Attendance,2017-06-25
2BD7879A13633D7966EFAF30285ACD97D7E4F31DC1316581C867A3CABBB00E9D,2017-08-22 23:01:00,First A&E Attendance,2017-08-22
987AC57102409F0AF5A6C3DED3E661AEFF4F19D294AFC4E34385093F1D6AB74D,2017-08-23 10:14:00,First A&E Attendance,2017-08-23
A4197B87D8D37C64E71CDAF4B4BC4D7265DBD544084B365DB22E17B405B68A86,2017-08-28 23:01:00,First A&E Attendance,2017-08-28
12DC2837BFCA3A379F508DA43C508DCC7E7F2F69744543303B87BF84E1048CA3,2017-08-30 08:00:00,First A&E Attendance,2017-08-30


In [100]:
unique(AE_filtered$attendance_category_Description)

[1] "First A&E Attendance"             "Follow Up Attendance - Planned"  
[3] "Unplanned - F Up Another ED"      "Unplanned - F Up This ED"        
[5] "Follow Up Attendance - Unplanned" "DOA"

In [101]:
AE_filtered |>
    filter(attendance_category_Description == 'DOA')

person_id,tbl_SecondaryCare_ae_nautilus_start_date,attendance_category_Description
<chr>,<chr>,<chr>
50233BFE7AA9F24DCBC570C4C244A30508E6AC112E883735CFA39CE992025CB4,2018-05-21 12:42:00,DOA


In [102]:
AE_filtered <- AE_filtered |>
    rename(description = attendance_category_Description) |>
    mutate(start_date = as.Date(tbl_SecondaryCare_ae_nautilus_start_date, format = "%Y-%m-%d %H:%M:%S"))

In [103]:
head(AE_filtered)

person_id,tbl_SecondaryCare_ae_nautilus_start_date,description,start_date
<chr>,<chr>,<chr>,<date>
5AAAC46458805915BE03E39E2EE548428C3427A64D919B30D6F9E42C70C45044,2021-10-10 04:01:00,First A&E Attendance,2021-10-10
AF26FF8CE455CC63945131643FD33864D7B7017FE2E171C89DCEC7D0B24BFCFD,2023-04-24 22:51:00,First A&E Attendance,2023-04-24
13E72B651A7665C776107E29C0475E0148ED3BF9023AAAF6B42C42086C5F4C64,2023-12-17 03:16:00,First A&E Attendance,2023-12-17
DF21EE69940BE6CD6ACAF925C723AD695D371F91B0D695B3C86A954E43DF6350,2021-12-06 19:03:00,First A&E Attendance,2021-12-06
526A9DD02760CA6A710A6A95FE76D797889BD3BFBAAEEDE5AD18868689EB4D81,2022-11-09 22:49:00,First A&E Attendance,2022-11-09
02134CA5E4A0C5B3CCBE26153D38614A1A3E87CD16ED88BBCE6A53628D8EA0A5,2024-08-09 16:58:00,First A&E Attendance,2024-08-09


In [106]:
AE_filtered_9899 <- AE_filtered |>
    semi_join(cohort_9899, by = "person_id")

In [108]:
AE_filtered_9899 |>
    arrange(start_date)

person_id,tbl_SecondaryCare_ae_nautilus_start_date,description,start_date
<chr>,<chr>,<chr>,<date>
FA8F3FE65CEDDE1CAADBFF0F235E53AECC56EFBDCE6DA24CBDBF88A1F6B35063,2017-01-26 09:03:00,First A&E Attendance,2017-01-26
2BD7879A13633D7966EFAF30285ACD97D7E4F31DC1316581C867A3CABBB00E9D,2017-08-22 23:01:00,First A&E Attendance,2017-08-22
A4197B87D8D37C64E71CDAF4B4BC4D7265DBD544084B365DB22E17B405B68A86,2017-08-28 23:01:00,First A&E Attendance,2017-08-28
6EC71241920766AF5E69CD64C1EA7D84A3CBB26048EA7696CE21867064A3B735,2017-09-18 14:00:00,First A&E Attendance,2017-09-18
740198D969C0A40663C28BB15A145A9953CDA3CB549CDA7F28AA249977042838,2017-09-22 07:00:00,First A&E Attendance,2017-09-22
11268124FD7DC7D3F23D2D111EE03C18D2E5E4DA25730AD1972F0B3D6A04AB40,2017-09-22 22:52:00,First A&E Attendance,2017-09-22
0FF858039D9299D1E0460A47298205231D52A959935AE1D6518A5CB361144FCD,2017-09-22 13:00:00,First A&E Attendance,2017-09-22
1F792FAB6FAC0DBB1920587C43291EEA35746B7222865048B95C36AC19DE2D8E,2017-09-23 22:15:00,First A&E Attendance,2017-09-23
9538601D513EA1AB7A8FDC522D3FB0CBB9D1918820EB364853AA7BCF63B5B498,2017-09-23 16:07:00,First A&E Attendance,2017-09-23


In [104]:
AE_filtered_9899 <- AE_filtered |>
    semi_join(cohort_9899, by = "person_id") |>
    filter(start_date < as.Date("2015-09-01")) |>
    select(-tbl_SecondaryCare_ae_nautilus_start_date)

In [105]:
AE_filtered_9899 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>


In [111]:
AE_filtered_9900 <- AE_filtered |>
    semi_join(cohort_9900, by = "person_id") 

In [109]:
AE_filtered_9900 <- AE_filtered |>
    semi_join(cohort_9900, by = "person_id") |>
    filter(start_date < as.Date("2016-09-01"))

In [112]:
AE_filtered_9900 |>
    arrange(start_date)

person_id,tbl_SecondaryCare_ae_nautilus_start_date,description,start_date
<chr>,<chr>,<chr>,<date>
987AC57102409F0AF5A6C3DED3E661AEFF4F19D294AFC4E34385093F1D6AB74D,2017-08-23 10:14:00,First A&E Attendance,2017-08-23
ADA171F70F6705DFB5C4CC4DC1F3D4F777D1EBD39AD6FBB43F874832CFEE3C9A,2017-09-22 17:43:00,First A&E Attendance,2017-09-22
27A1BFC4DAB6A9753E6D66E746259DBA1BE903D13CA1D8D79334D40284A019A7,2017-09-23 19:05:00,First A&E Attendance,2017-09-23
4BC61F8975ABBD71D532B841CF16265957F5F453C37240D3BA25BB7A92601368,2017-09-24 10:00:00,First A&E Attendance,2017-09-24
3F01C0C4497816A2391716FDA15DA1BC577A4E8B272453BC325459C6819DFED2,2017-09-24 01:03:00,First A&E Attendance,2017-09-24
71C2445B16276AFCBEE2F6D2731F6459CF2D40D55D5941D3B8218DE70A987EDF,2017-09-25 22:49:00,First A&E Attendance,2017-09-25
8703E6AF05916E33A89797A0EC21128D1853D448E29D6C2C2D9D0F298817C275,2017-09-25 00:44:00,First A&E Attendance,2017-09-25
162EBF3A92B1EC9A158AD19C34CC83177C583B6E18A92889B378D88E506EAEEA,2017-09-25 20:57:00,First A&E Attendance,2017-09-25
05A567CB36220A5A84ED3358E7FCDB003D3F1FF93A77A70930E3E2CD951F9E91,2017-09-25 14:00:00,First A&E Attendance,2017-09-25


In [110]:
AE_filtered_9900 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>


In [63]:
inpatient_filtered_0001 <- inpatient_filtered |>
    semi_join(cohort_0001, by = "person_id") |>
    filter(stay_start_date < as.Date("2017-09-01"))

In [64]:
inpatient_filtered_0001 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,1256
2,463
3,164
4,79
5,41
6,29
7,12
8,10
9,10


In [65]:
inpatient_filtered_0102 <- inpatient_filtered |>
    semi_join(cohort_0102, by = "person_id") |>
    filter(stay_start_date < as.Date("2018-09-01"))

In [66]:
inpatient_filtered_0102 |>
  count(person_id) |>       
  count(n) |>               
  rename(frequency = n, count = nn)

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.


frequency,count
<int>,<int>
1,1362
2,470
3,210
4,85
5,49
6,20
7,13
8,9
9,11


In [67]:
inpatient_stays <- rbind(inpatient_filtered_9899, inpatient_filtered_9900, inpatient_filtered_0001, inpatient_filtered_0102)

In [68]:
head(inpatient_stays)

person_id,stay_start_date,length_of_stay
<chr>,<chr>,<dbl>
92D116B37F94927C3909E3A0DEC93313C6E9DC8AE59A74E3741A4EC31BE423F6,2006-12-30,2
B9BA8CA19EB91125A2D67B96A3AD8C7FAFB3B9400C431D270F1747A5B2CB3021,2007-01-01,2
93C9C7B8080DC785358A03AA02A6F16621203AE0F5D46DAA931300E0598E15CF,2007-01-02,2
4D7E7CCDF862ADAE8CF68EA1EFDCD2DF51FA3922CF10F3E547B611FEFF04D0BA,2007-01-03,2
B38B8FA67AB9E90BAFF692608643CBD90DFBEBED10400C09F2FBE16BC0C8E726,2007-01-04,2
1625A5EC5A3C4CEC388E1031C1DD98A75B05C2AB8B14C2643B5A6C70878DB23D,2007-01-04,2
